## Tratamento Inicial

In [ ]:
import pandas as pd
df = pd.read_csv('./dataset/trabalho.csv', sep=';')

In [5]:
print(df['Município'].unique())  # Lista todos os municípios

['Cubatão - SP' 'Ribeirão Pires - SP' 'Catanduva - SP' 'Paulínia - SP'
 'Leme - SP' 'Ourinhos - SP' 'Poá - SP' 'Assis - SP' 'Itanhaém - SP']


In [13]:
from chardet import detect

# Detectar encoding automaticamente
with open('./dataset/trabalho.csv', 'rb') as f:
    encoding = detect(f.read())['encoding']

try:
    df = pd.read_csv('./dataset/trabalho.csv', sep=None, engine='python', encoding=encoding)
    print("Arquivo lido com sucesso!")
    print(f"Encoding detectado: {encoding}")
    print(f"Delimitador detectado: {df._engine.delimiter}")
except Exception as e:
    print(f"Erro crítico: {e}")

Arquivo lido com sucesso!
Encoding detectado: UTF-8-SIG
Erro crítico: 'DataFrame' object has no attribute '_engine'


## Verificar as Cidades do CSV

In [14]:
from chardet import detect

# 1. Função para detectar delimitador
def detectar_delimitador(arquivo):
    with open(arquivo, 'r', encoding='utf-8-sig') as f:
        primeira_linha = f.readline()
    delimitadores = [',', ';', '\t']
    return max(delimitadores, key=lambda d: primeira_linha.count(d))

# 2. Carregar o CSV
try:
    delimitador = detectar_delimitador('./dataset/trabalho.csv')
    df = pd.read_csv('./dataset/trabalho.csv', sep=delimitador, encoding='utf-8-sig')
    print("✅ Arquivo carregado com sucesso!")
    print(f"Delimitador detectado: '{delimitador}'")
    
except Exception as e:
    print(f"❌ Erro ao ler arquivo: {e}")
    exit()

# 3. Função de processamento (agora verificando se o município existe)
def processar_municipio(df, municipio):
    if municipio not in df['Município'].values:
        print(f"\nMunicípio '{municipio}' não encontrado. Opções disponíveis:")
        print(df['Município'].unique())
        return
    
    df_mun = df[df['Município'] == municipio]
    anos = range(2008, 2025, 4)
    
    for ano in anos:
        df_ano = df_mun[df_mun['Ano'] == ano]
        if df_ano.empty:
            print(f"\nNão há dados para {municipio} em {ano}")
            continue
            
        # Processamento dos dados
        top5 = df_ano.groupby('Código SH2')['Valor US$ FOB'].sum().nlargest(5).reset_index()
        total = df_ano['Valor US$ FOB'].sum()
        
        # Formatação da saída
        print(f"\n📊 {municipio} - Ano {ano}")
        print("=" * 50)
        print(f"{'Seção':<8} {'Valor (US$)':>15} {'%':>8}")
        print("-" * 50)
        
        for _, row in top5.iterrows():
            percent = (row['Valor US$ FOB'] / total) * 100
            print(f"{row['Código SH2']:<8} {row['Valor US$ FOB']:>15,.2f} {percent:>7.1f}%")
        
        outros = total - top5['Valor US$ FOB'].sum()
        print("-" * 50)
        print(f"{'Subtotal':<8} {top5['Valor US$ FOB'].sum():>15,.2f} {(100 - (outros/total)*100):>7.1f}%")
        print(f"{'Outros':<8} {outros:>15,.2f} {(outros/total)*100:>7.1f}%")
        print(f"{'Total':<8} {total:>15,.2f} {100:>7.0f}%")

# 4. Execução
processar_municipio(df, "Assis - SP")


✅ Arquivo carregado com sucesso!
Delimitador detectado: ';'

Município 'Assis - SP' não encontrado. Opções disponíveis:
['\rCubatão - SP' '\rRibeirão Pires - SP' '\rCatanduva - SP'
 '\rPaulínia - SP' '\rLeme - SP' '\rOurinhos - SP' '\rPoá - SP'
 '\rAssis - SP' '\rItanhaém - SP']


## Gerar os Dados

In [ ]:
# Carregar e limpar os dados
try:
    df = pd.read_csv('./dataset/trabalho.csv', sep=';', encoding='utf-8-sig')
    df = df.apply(lambda x: x.str.strip('\r ') if x.dtype == 'object' else x)
    print("✅ Arquivo carregado com sucesso!")
except Exception as e:
    print(f"❌ Erro ao ler arquivo: {e}")
    exit()

def analisar_fluxo(df, municipio, fluxo):
    # Filtrar por município e fluxo
    df_filtrado = df[(df['Município'] == municipio) & (df['Fluxo'] == fluxo)]
    
    if df_filtrado.empty:
        print(f"\nNão há dados de {fluxo} para {municipio}")
        return
    
    anos = range(2008, 2025, 4)
    
    for ano in anos:
        df_ano = df_filtrado[df_filtrado['Ano'] == ano]
        if df_ano.empty:
            print(f"\nNão há dados de {fluxo} para {municipio} em {ano}")
            continue
            
        top5 = df_ano.groupby('Código SH2')['Valor US$ FOB'].sum().nlargest(5).reset_index()
        total = df_ano['Valor US$ FOB'].sum()
        
        print(f"\n📊 {fluxo} - {municipio} - Ano {ano}")
        print("=" * 60)
        print(f"{'Seção':<8} {'Valor (US$)':>20} {'%':>10}")
        print("-" * 60)
        
        for _, row in top5.iterrows():
            percent = (row['Valor US$ FOB'] / total) * 100
            print(f"{row['Código SH2']:<8} {row['Valor US$ FOB']:>20,.2f} {percent:>9.1f}%")
        
        outros = total - top5['Valor US$ FOB'].sum()
        percent_subtotal = (top5['Valor US$ FOB'].sum() / total) * 100
        percent_outros = (outros / total) * 100
        
        print("-" * 60)
        print(f"{'Subtotal':<8} {top5['Valor US$ FOB'].sum():>20,.2f} {percent_subtotal:>9.1f}%")
        print(f"{'Outros':<8} {outros:>20,.2f} {percent_outros:>9.1f}%")
        print(f"{'Total':<8} {total:>20,.2f} {100:>9.0f}%")

# Processar para um município específico
municipio = "Ourinhos - SP"

print("\n" + "="*80)
print(f"ANÁLISE COMPLETA PARA: {municipio.upper()}")
print("="*80)

# Análise de Exportação
print("\n🔵 EXPORTAÇÃO 🔵")
analisar_fluxo(df, municipio, "Exportação")

# Análise de Importação
print("\n🔴 IMPORTAÇÃO 🔴")
analisar_fluxo(df, municipio, "Importação")

# Para ver todos os municípios limpos:
print("\n🔍 Municípios disponíveis:")
print([m for m in df['Município'].str.strip().unique() if not pd.isna(m)])

✅ Arquivo carregado com sucesso!

ANÁLISE COMPLETA PARA: OURINHOS - SP

🔵 EXPORTAÇÃO 🔵

📊 Exportação - Ourinhos - SP - Ano 2008
Seção             Valor (US$)          %
------------------------------------------------------------
12              24,973,676.00      55.7%
10               7,218,657.00      16.1%
84               7,104,189.00      15.9%
17               4,268,850.00       9.5%
9                  702,522.00       1.6%
------------------------------------------------------------
Subtotal        44,267,894.00      98.8%
Outros             531,085.00       1.2%
Total           44,798,979.00       100%

📊 Exportação - Ourinhos - SP - Ano 2012
Seção             Valor (US$)          %
------------------------------------------------------------
12              23,873,201.00      36.7%
84              20,949,559.00      32.2%
17              10,986,037.00      16.9%
10               5,655,107.00       8.7%
73               1,375,881.00       2.1%
---------------------------------

# Código para Exportar em Excel

In [7]:

# 1. Função para processar os dados (igual à anterior)
def analisar_fluxo_para_excel(df, municipio, fluxo):
    resultados = []
    df_filtrado = df[(df['Município'] == municipio) & (df['Fluxo'] == fluxo)]
    
    if df_filtrado.empty:
        print(f"\nNão há dados de {fluxo} para {municipio}")
        return pd.DataFrame()

    anos = sorted(df_filtrado['Ano'].unique())  # Pega todos os anos disponíveis
    
    for ano in anos:
        df_ano = df_filtrado[df_filtrado['Ano'] == ano]
        if df_ano.empty:
            continue
            
        top5 = df_ano.groupby('Código SH2')['Valor US$ FOB'].sum().nlargest(5).reset_index()
        total = df_ano['Valor US$ FOB'].sum()
        
        for _, row in top5.iterrows():
            percent = (row['Valor US$ FOB'] / total) * 100
            resultados.append({
                'Município': municipio,
                'Fluxo': fluxo,
                'Ano': ano,
                'Seção': row['Código SH2'],
                'Valor (US$)': row['Valor US$ FOB'],
                '% do Total': percent,
                'Tipo': 'Top 5'
            })
        
        outros = total - top5['Valor US$ FOB'].sum()
        percent_outros = (outros / total) * 100
        
        resultados.extend([
            {
                'Município': municipio,
                'Fluxo': fluxo,
                'Ano': ano,
                'Seção': 'Subtotal',
                'Valor (US$)': top5['Valor US$ FOB'].sum(),
                '% do Total': 100 - percent_outros,
                'Tipo': 'Total'
            },
            {
                'Município': municipio,
                'Fluxo': fluxo,
                'Ano': ano,
                'Seção': 'Outros',
                'Valor (US$)': outros,
                '% do Total': percent_outros,
                'Tipo': 'Total'
            },
            {
                'Município': municipio,
                'Fluxo': fluxo,
                'Ano': ano,
                'Seção': 'TOTAL',
                'Valor (US$)': total,
                '% do Total': 100,
                'Tipo': 'Total'
            }
        ])
    
    return pd.DataFrame(resultados)

# 2. Carregar os dados originais
try:
    df = pd.read_csv('./dataset/trabalho.csv', sep=';', encoding='utf-8-sig')
    df = df.apply(lambda x: x.str.strip('\r ') if x.dtype == 'object' else x)
    print("✅ Arquivo carregado com sucesso!")
except Exception as e:
    print(f"❌ Erro ao ler arquivo: {e}")
    exit()

# 3. Processar os dados
municipio = "Ribeirão Pires - SP"
df_export = analisar_fluxo_para_excel(df, municipio, "Exportação")
df_import = analisar_fluxo_para_excel(df, municipio, "Importação")

# 4. Exportar para Excel (com abas separadas)
if not df_export.empty or not df_import.empty:
    nome_excel = f"comercio_exterior_{municipio.lower().replace(' - ', '_').replace(' ', '_')}.xlsx"
    
    with pd.ExcelWriter(nome_excel, engine='openpyxl') as writer:
        if not df_export.empty:
            df_export.to_excel(writer, sheet_name='Exportação', index=False)
        if not df_import.empty:
            df_import.to_excel(writer, sheet_name='Importação', index=False)
    
    print(f"✅ Arquivo Excel salvo em: {nome_excel}")

# 5. Opcional: Mostrar preview no console
if not df_export.empty:
    print("\nPreview dos dados de exportação:")
    print(df_export.head())
if not df_import.empty:
    print("\nPreview dos dados de importação:")
    print(df_import.head())

✅ Arquivo carregado com sucesso!
✅ Arquivo Excel salvo em: comercio_exterior_ribeirão_pires_sp.xlsx

Preview dos dados de exportação:
             Município       Fluxo   Ano Seção  Valor (US$)  % do Total   Tipo
0  Ribeirão Pires - SP  Exportação  2008    93     97536971   78.839840  Top 5
1  Ribeirão Pires - SP  Exportação  2008    73      7285356    5.888806  Top 5
2  Ribeirão Pires - SP  Exportação  2008    85      6882192    5.562926  Top 5
3  Ribeirão Pires - SP  Exportação  2008    84      4285405    3.463924  Top 5
4  Ribeirão Pires - SP  Exportação  2008    36      3062803    2.475686  Top 5

Preview dos dados de importação:
             Município       Fluxo   Ano Seção  Valor (US$)  % do Total   Tipo
0  Ribeirão Pires - SP  Importação  2008    84      8049208   18.772603  Top 5
1  Ribeirão Pires - SP  Importação  2008    93      4956750   11.560280  Top 5
2  Ribeirão Pires - SP  Importação  2008    72      4253107    9.919223  Top 5
3  Ribeirão Pires - SP  Importação  2008  

In [30]:
# Para ver todos os municípios limpos:
print("\n🔍 Municípios disponíveis:")
print([m for m in df['Município'].str.strip().unique() if not pd.isna(m)])


🔍 Municípios disponíveis:
['Cubatão - SP', 'Ribeirão Pires - SP', 'Catanduva - SP', 'Paulínia - SP', 'Leme - SP', 'Ourinhos - SP', 'Poá - SP', 'Assis - SP', 'Itanhaém - SP']
